In [ ]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2019-10.csv.gz

In [ ]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv

In [5]:
import pandas as pd

In [23]:
file_path = 'green_tripdata_2019-10.csv.gz'
df = pd.read_csv(file_path)
df.head()

/tmp/ipykernel_24079/1014065643.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2.0,2019-10-01 00:26:02,2019-10-01 00:39:58,N,1.0,112,196,1.0,5.88,18.0,0.50,0.5,0.00,0.0,NaN,0.3,19.30,2.0,1.0,0.0
1,1.0,2019-10-01 00:18:11,2019-10-01 00:22:38,N,1.0,43,263,1.0,0.80,5.0,3.25,0.5,0.00,0.0,NaN,0.3,9.05,2.0,1.0,0.0
2,1.0,2019-10-01 00:09:31,2019-10-01 00:24:47,N,1.0,255,228,2.0,7.50,21.5,0.50,0.5,0.00,0.0,NaN,0.3,22.80,2.0,1.0,0.0
3,1.0,2019-10-01 00:37:40,2019-10-01 00:41:49,N,1.0,181,181,1.0,0.90,5.5,0.50,0.5,0.00,0.0,NaN,0.3,6.80,2.0,1.0,0.0
4,2.0,2019-10-01 00:08:13,2019-10-01 00:17:56,N,1.0,97,188,1.0,2.52,10.0,0.50,0.5,2.26,0.0,NaN,0.3,13.56,1.0,1.0,0.0


In [24]:
taxi_zone_lookup_path = 'taxi_zone_lookup.csv'
taxi_zone_lookup = pd.read_csv(taxi_zone_lookup_path)
taxi_zone_lookup.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [25]:
df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])
df['lpep_pickup_datetime'].dtype


dtype('<M8[ns]')

In [26]:
# Define the date range: from October 1, 2019 to November 1, 2019 (exclusive)
start_date = '2019-10-01'
end_date = '2019-11-01'

# Filter trips that occurred in this date range
filtered_df = df[(df['lpep_pickup_datetime'] >= start_date) & (df['lpep_pickup_datetime'] < end_date)]


In [16]:
df = df.dropna(subset=[distance_column])


In [27]:
# Define the column for trip distance
distance_column = 'trip_distance'

# Filter and count trips for each distance range
up_to_1_mile = len(filtered_df[filtered_df[distance_column] <= 1])
between_1_and_3_miles = len(filtered_df[(filtered_df[distance_column] > 1) & (filtered_df[distance_column] <= 3)])
between_3_and_7_miles = len(filtered_df[(filtered_df[distance_column] > 3) & (filtered_df[distance_column] <= 7)])
between_7_and_10_miles = len(filtered_df[(filtered_df[distance_column] > 7) & (filtered_df[distance_column] <= 10)])
over_10_miles = len(filtered_df[filtered_df[distance_column] > 10])

# Print the results
print(f"Trips up to 1 mile: {up_to_1_mile}")
print(f"Trips between 1 and 3 miles: {between_1_and_3_miles}")
print(f"Trips between 3 and 7 miles: {between_3_and_7_miles}")
print(f"Trips between 7 and 10 miles: {between_7_and_10_miles}")
print(f"Trips over 10 miles: {over_10_miles}")


Trips up to 1 mile: 104830
Trips between 1 and 3 miles: 198995
Trips between 3 and 7 miles: 109642
Trips between 7 and 10 miles: 27686
Trips over 10 miles: 35201


In [43]:
max_row = df.loc[df['trip_distance'].idxmax()]
print(max_row['lpep_pickup_datetime'])

2019-10-31 23:23:41


In [30]:
date_filtered_df = df[df['lpep_pickup_datetime'].dt.date == pd.to_datetime('2019-10-18').date()]


In [35]:
# Assuming 'pickup_location_id' is the common column to merge on
merged_df = pd.merge(date_filtered_df, taxi_zone_lookup, left_on='PULocationID', right_on='LocationID', how='left')

# Display the first few rows of the merged DataFrame
merged_df.head()


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,LocationID,Borough,Zone,service_zone
0,2.0,2019-10-18 23:35:11,2019-10-19 00:09:42,N,1.0,247,48,1.0,6.34,21.0,...,NaN,0.3,22.30,2.0,1.0,0.0,247,Bronx,West Concourse,Boro Zone
1,2.0,2019-10-18 00:00:40,2019-10-18 00:04:41,N,1.0,145,112,5.0,0.84,5.5,...,NaN,0.3,8.16,1.0,1.0,0.0,145,Queens,Long Island City/Hunters Point,Boro Zone
2,2.0,2019-10-18 00:00:10,2019-10-18 00:13:16,N,1.0,95,215,1.0,3.38,13.0,...,NaN,0.3,14.30,2.0,1.0,0.0,95,Queens,Forest Hills,Boro Zone
3,2.0,2019-10-18 00:01:17,2019-10-18 00:09:42,N,1.0,82,82,1.0,1.67,8.0,...,NaN,0.3,9.30,2.0,1.0,0.0,82,Queens,Elmhurst,Boro Zone
4,2.0,2019-10-18 22:58:52,2019-10-19 00:07:30,N,1.0,74,116,1.0,7.69,31.5,...,NaN,0.3,39.36,1.0,1.0,0.0,74,Manhattan,East Harlem North,Boro Zone


In [42]:
pickup_totals = merged_df.groupby(['PULocationID', 'Zone'])['total_amount'].sum()
top_pickups = pickup_totals[pickup_totals > 13000]

# Sort the results in descending order
top_pickups = top_pickups.sort_values(ascending=False)

# Display the top pickup locations
top_pickups


PULocationID  Zone               
74            East Harlem North      18686.68
75            East Harlem South      16797.26
166           Morningside Heights    13029.79
Name: total_amount, dtype: float64

In [52]:
merged_all_df = pd.merge(df, taxi_zone_lookup, left_on='PULocationID', right_on='LocationID', how='left')

east_harlem_north_data = merged_all_df[(merged_all_df['Zone'] == "East Harlem North")]

largest_tip_row = east_harlem_north_data.loc[east_harlem_north_data['tip_amount'].idxmax()]

# Display the relevant details: pickup time, tip amount, and zone
largest_tip_row[['lpep_pickup_datetime', 'tip_amount', 'Zone']]


lpep_pickup_datetime    2019-10-25 15:50:05
tip_amount                             87.3
Zone                      East Harlem North
Name: 308891, dtype: object